# Kata 14 — Few-Shot para Bordes

> Spec: [`specs/014-few-shot-calibration/spec.md`](../../specs/014-few-shot-calibration/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=8)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cuando la tarea es subjetiva (tono, formato no estándar, juicio estético), 2-4 ejemplos input/output desplazan la distribución del modelo más rápido y más barato que un párrafo de instrucciones zero-shot.

## 2. Por qué importa

"Responde en estilo casual chileno" en prosa es ambiguo; mostrar 3 ejemplos de cómo se ve lo elimina.

## 3. Ejemplo correcto — clasificación con 3 ejemplos de bordes

In [ ]:
import json

CLASSIFY_TOOL = {
    "name": "classify_ticket",
    "description": "Clasifica el ticket de soporte.",
    "input_schema": {
        "type": "object",
        "properties": {
            "category": {"type": "string", "enum": ["billing","auth","performance","feature","other"]},
            "urgency": {"type": "string", "enum": ["low","medium","high"]},
        },
        "required": ["category","urgency"],
    },
}

EXAMPLES = '''ejemplo 1
ticket: "no me llega la factura desde hace 3 meses, ya pagué"
classify_ticket(category="billing", urgency="high")

ejemplo 2
ticket: "tengo una idea cool pa mejorar el onboarding cuando puedan"
classify_ticket(category="feature", urgency="low")

ejemplo 3
ticket: "el sistema responde pero raro, como pegado, no es del todo lento"
classify_ticket(category="performance", urgency="medium")
'''

EDGE_CASE = "uy onda no me deja entrar y tengo demo en 1h, ayuda urgente porfa"

def classify(client, *, system, ticket):
    resp = client.messages.create(
        system=system,
        messages=[{"role":"user","content": ticket}],
        tools=[CLASSIFY_TOOL],
        tool_choice={"type":"tool","name":"classify_ticket"},
    )
    return next(b.input for b in resp.content if b.type == "tool_use")

system_few = "Clasifica tickets como en los ejemplos:\n\n" + EXAMPLES
result_few = classify(client, system=system_few, ticket=EDGE_CASE)
print("few-shot:", result_few)

## 4. Anti-patrón — zero-shot puro

In [ ]:
system_zero = "Clasifica el ticket en una de las categorías estándar."
result_zero = classify(client, system=system_zero, ticket=EDGE_CASE)
print("zero-shot:", result_zero)

En zero-shot el modelo tiende a categorías "seguras" (`other` con urgencia `medium`) o malinterpreta jergas. Los ejemplos calibran la distribución hacia la operación real.

### 4.1 Sobre-ajuste con demasiados ejemplos

In [ ]:
too_many = EXAMPLES + "\n".join(f"ejemplo {i}\nticket: 'nada importante {i}'\nclassify_ticket(category='other', urgency='low')" for i in range(4, 25))
result_overflow = classify(client, system="Clasifica como en los ejemplos:\n\n" + too_many, ticket=EDGE_CASE)
print("over-fit (25 ejemplos):", result_overflow)

Con 25 ejemplos en su mayoría triviales, el modelo se sesga al patrón "other/low" y deja de capturar urgencias reales. Más no es mejor.

## 5. Argumento de certificación

- **2-4 ejemplos** que cubran los **bordes** del dominio, no el caso fácil.
- Cada ejemplo en el **mismo schema** que la salida esperada.
- **Few-shot complementa** al schema (Kata 5), no lo reemplaza.
- **Conexión con otros katas**: los ejemplos son contenido estable → caché-friendly (Kata 10), van al inicio del system prompt para alta atención (Kata 11).

## 6. Auto-evaluación

**1. ¿Cuándo añadir un ejemplo más empeora el resultado?**

Cuando satura la atención (más allá de ~4-5 para muchos casos) o introduce contradicciones con los anteriores. La curva de mejora es cóncava: los primeros 2-3 valen oro; el quinto rara vez compensa.

**2. ¿Por qué los ejemplos van al inicio (estático) y no al final?**

Porque son contenido estable: van en el bloque cacheable (Kata 10) y en zona de alta atención (Kata 11). Si los pongo al final, los pierdo del caché y compiten con la pregunta del usuario.

**3. Si los ejemplos contradicen el schema, ¿quién gana?**

El schema. `tool_choice` con `input_schema` rechaza valores fuera del enum aunque el ejemplo los muestre. Por eso los ejemplos deben ser consistentes con el schema; un ejemplo que use una categoría no listada degrada los resultados.